In [36]:
# === IMPORT CONFIGURATION ===
import os
from pathlib import Path

# Import settings from config
from config import (
    BASE_DIR, DATA_DIR, LOGS_DIR, TRANSCRIPTS_DIR, RAW_DIR,
    OUTPUT_DIR, CLEANED_TRANSCRIPTS
)

# Define additional paths specific to this notebook
SENTIMENT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "enhanced_sentiment_viz")

# Make sure our output folder exists before we try to save things there
os.makedirs(SENTIMENT_OUTPUT_DIR, exist_ok=True)

# Import all the packages we'll need for this analysis
import sys
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from datetime import datetime

# Download NLTK resources if we don't have them yet - we need these for text analysis
# The 'quiet=True' just keeps it from printing a bunch of download messages
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print("Performing enhanced sentiment analysis...")

# Let's double check we're in the right place and our data file exists
print(f"Current working directory: {os.getcwd()}")
print(f"Cleaned transcripts file exists: {os.path.exists(CLEANED_TRANSCRIPTS)}")

# ===== FILE PATH HANDLING =====
# Since our data might be in different locations depending on how previous notebooks ran,
# I'm going to look in several places and use the most recent file
file_paths = {
    "cleaned_transcripts": CLEANED_TRANSCRIPTS,
    "main_8k": os.path.join(DATA_DIR, "cleaned_8k_data.csv"),
    "output_8k": os.path.join(OUTPUT_DIR, "cleaned_8k_data.csv")
}

# Check which files exist and when they were last modified - we'll use the newest one
existing_files = {}
for location, path in file_paths.items():
    if os.path.exists(path):
        mod_time = os.path.getmtime(path)
        existing_files[location] = {
            "path": path,
            "mod_time": mod_time,
            "date": datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
        }

if not existing_files:
    raise FileNotFoundError("Could not find any cleaned data files - make sure you've run the data cleaning notebook first")

# Use the most recently modified file to ensure we have the latest data
if len(existing_files) > 1:
    most_recent = max(existing_files.items(), key=lambda x: x[1]["mod_time"])
    file_to_use = most_recent[1]["path"]
    print(f"Multiple data files found. Using most recent: {file_to_use} (modified {most_recent[1]['date']})")
else:
    location = list(existing_files.keys())[0]
    file_to_use = existing_files[location]["path"]
    print(f"Using data file: {file_to_use} (modified {existing_files[location]['date']})")

# Load our data into a DataFrame
df = pd.read_csv(file_to_use)

# Let's see what companies we found
print(f"Found {len(df)} records with {df['ticker'].nunique()} companies")
print(f"Companies in dataset: {sorted(df['ticker'].unique())}")

# 1. CREATE A FINANCE-SPECIFIC WORD LIST
# Here I'm creating custom dictionaries of finance terms with their sentiment scores
# These are words that are specifically positive in earnings call contexts
# The numbers represent sentiment strength (higher = more positive)
financial_pos = {
    'beat': 3.0, 'exceeded': 3.0, 'growth': 2.0, 'grew': 2.0, 'increase': 1.5, 
    'increased': 1.5, 'expanding': 1.5, 'expanded': 1.5, 'profitable': 2.0, 
    'profitability': 2.0, 'margin': 0.5, 'strong': 2.0, 'strength': 2.0, 
    'opportunity': 1.5, 'opportunities': 1.5, 'outperform': 2.5, 'outperformed': 2.5,
    'record': 2.0, 'robust': 2.0, 'higher': 1.0, 'innovation': 1.5, 'innovative': 1.5,
    'succeeded': 2.0, 'success': 2.0, 'successful': 2.0, 'positive': 1.5, 
    'achieve': 1.0, 'achieved': 1.0, 'achievement': 1.0, 'improving': 1.0,
    'improved': 1.0, 'improvement': 1.0, 'leadership': 1.0, 'leading': 1.0,
    'advantage': 1.5, 'advantages': 1.5, 'favorable': 1.5
}

# Words that are negative in earnings calls - the numbers are negative to indicate bad sentiment
# Bigger negative numbers = stronger negative sentiment
financial_neg = {
    'miss': -2.0, 'missed': -2.0, 'decline': -2.0, 'declined': -2.0, 
    'decrease': -1.5, 'decreased': -1.5, 'shrinking': -1.5, 'shrunk': -1.5,
    'loss': -2.0, 'losses': -2.0, 'losing': -1.5, 'lost': -1.5, 'weak': -2.0, 
    'weakness': -2.0, 'challenge': -1.0, 'challenges': -1.0, 'challenging': -1.0,
    'underperform': -2.5, 'underperformed': -2.5, 'disappointing': -2.0, 
    'disappointed': -2.0, 'disappointment': -2.0, 'lower': -1.0, 'slower': -1.0,
    'slowdown': -1.5, 'negative': -1.5, 'failed': -2.0, 'failure': -2.0,
    'downturn': -2.0, 'declining': -1.5, 'deteriorating': -2.0, 'deteriorated': -2.0,
    'deterioration': -2.0, 'adversely': -1.5, 'adverse': -1.5, 'unfavorable': -1.5,
    'litigation': -1.0, 'regulatory': -0.5, 'investigation': -1.5, 'penalty': -2.0,
    'penalties': -2.0, 'recession': -2.5, 'crisis': -2.5, 'threat': -1.5,
    'difficult': -1.0, 'disruption': -1.5, 'disruptions': -1.5, 'restructuring': -1.0,
    'layoff': -2.0, 'layoffs': -2.0, 'delay': -1.0, 'delayed': -1.0, 'delays': -1.0,
    'postpone': -1.0, 'postponed': -1.0, 'suspended': -2.0, 'suspend': -2.0,
    'termination': -2.0, 'terminate': -2.0, 'terminated': -2.0
}

# Uncertainty words are usually negative for stocks - investors hate uncertainty!
# These tend to signal hesitation or lack of confidence
uncertainty_terms = {
    'may': -0.5, 'might': -0.5, 'could': -0.5, 'possibly': -0.5, 'uncertain': -1.0,
    'uncertainty': -1.0, 'risk': -1.0, 'risks': -1.0, 'exposed': -0.5, 'exposure': -0.5,
    'volatile': -1.0, 'volatility': -1.0, 'cautious': -0.5, 'caution': -0.5,
    'warning': -1.0, 'warn': -1.0, 'warned': -1.0, 'concerned': -1.0, 'concern': -1.0,
    'concerns': -1.0, 'hesitant': -0.5, 'conditional': -0.5
}

# Put all our financial words together in one big dictionary we can use for analysis
finance_lexicon = {**financial_pos, **financial_neg, **uncertainty_terms}

# 2. CLEAN AND PREP THE TEXT
def preprocess_text(text):
    """This function cleans up text to make it ready for analysis"""
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Get rid of HTML tags that might be in the text
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove numbers and special characters - we're just focused on words
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Clean up any extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Return everything in lowercase for consistency
    return text.lower()

# This function finds the most positive and negative sentences in a text
# Useful for understanding what's driving sentiment scores
def extract_extreme_sentences(text, sia):
    """Find the most positive and negative sentences in the text"""
    if pd.isna(text) or not isinstance(text, str) or len(text) < 20:
        return {"most_positive": "", "most_negative": ""}

    # Break the text into sentences (this regex handles different punctuation)
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)

    # Ignore very short sentences - they don't usually contain meaningful sentiment
    sentences = [s for s in sentences if len(s) > 15]

    if not sentences:
        return {"most_positive": "", "most_negative": ""}

    # Calculate sentiment for each sentence using our analyzer
    sentiments = []
    for sentence in sentences:
        sentiment = sia.polarity_scores(sentence)
        sentiments.append((sentence, sentiment['compound']))

    # Find the extreme sentences by sorting
    sentiments.sort(key=lambda x: x[1])
    most_negative = sentiments[0][0] if sentiments else ""
    most_positive = sentiments[-1][0] if sentiments else ""

    return {"most_positive": most_positive, "most_negative": most_negative}

# Create a column for our processed text
df['processed_text'] = ""

# Figure out which text columns we have available to analyze
available_text_columns = [col for col in df.columns if col in 
                          ['outlook_cleaned', 'product_mentions_cleaned', 'transcript_text', 'cleaned_text']]
print(f"Available text columns for analysis: {available_text_columns}")

# Start with the most useful columns first - outlook and product mentions are usually 
# the most informative parts of earnings calls
for text_col in available_text_columns:
    if text_col in ['outlook_cleaned', 'product_mentions_cleaned']:  # These are often most informative
        df['processed_text'] += " " + df[text_col].apply(preprocess_text)

# If we don't have specialized columns, just use whatever text we can find
if df['processed_text'].str.strip().eq('').all() and available_text_columns:
    print("No specialized text columns found. Using available text data.")
    for text_col in available_text_columns:
        df['processed_text'] += " " + df[text_col].apply(preprocess_text)

# Set up our sentiment analyzer - VADER is a good one for social media type text
sia = SentimentIntensityAnalyzer()

# Add our custom finance words to the analyzer's lexicon so it recognizes them
for word, score in finance_lexicon.items():
    sia.lexicon[word] = score

# 3. CALCULATE SENTIMENT IN MULTIPLE WAYS
# We'll store all our results in this list then convert to a DataFrame later
sentiment_results = []

# Go through each row in our data
for idx, row in df.iterrows():
    text = row['processed_text']
    ticker = row['ticker']

    # Some rows might not have filing dates
    filing_date = row.get('filing_date') if 'filing_date' in row else None

    # Handle cases where we don't have any text to analyze
    if not text or len(text) < 20:
        # Even with no text, keep the record with neutral sentiment
        sentiment_results.append({
            'ticker': ticker,
            'filing_date': filing_date,
            'sentiment_compound': 0,
            'sentiment_positive': 0,
            'sentiment_negative': 0,
            'sentiment_neutral': 0,
            'financial_sentiment_score': 0,
            'most_positive_sentence': "",
            'most_negative_sentence': ""
        })
        continue

    # Get standard VADER sentiment scores 
    sentiment = sia.polarity_scores(text)

    # Now calculate our custom financial sentiment that's more sensitive to financial terms
    words = word_tokenize(text.lower())

    # Remove stop words (common words like "the", "and", etc.) that don't carry sentiment
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in words if word not in stop_words]

    # Count and score the financial terms
    financial_words_score = 0
    financial_words_count = 0

    for word in filtered_words:
        if word in finance_lexicon:
            financial_words_score += finance_lexicon[word]
            financial_words_count += 1

    # Calculate our financial sentiment score
    financial_sentiment_score = 0
    if financial_words_count > 0:
        # Scale to spread out the values more than standard sentiment
        # This gives us more differentiation between slightly positive and very positive
        financial_sentiment_score = financial_words_score / (financial_words_count * 2)

    # Find the most positive and negative sentences for display
    extreme_sentences = extract_extreme_sentences(text, sia)

    # Store all the sentiment metrics for this row
    sentiment_results.append({
        'ticker': ticker,
        'filing_date': filing_date,
        'sentiment_compound': sentiment['compound'],
        'sentiment_positive': sentiment['pos'],
        'sentiment_negative': sentiment['neg'],
        'sentiment_neutral': sentiment['neu'],
        'financial_sentiment_score': financial_sentiment_score,
        'most_positive_sentence': extreme_sentences['most_positive'],
        'most_negative_sentence': extreme_sentences['most_negative']
    })

# Convert our results to a DataFrame
sentiment_df = pd.DataFrame(sentiment_results)

# Make sure we have all companies - check if any are missing
all_companies = set(df['ticker'].unique())
processed_companies = set(sentiment_df['ticker'].unique())
missing_companies = all_companies - processed_companies

if missing_companies:
    print(f"WARNING: {len(missing_companies)} companies missing from sentiment analysis: {missing_companies}")
    # Add dummy entries for missing companies so they show up in visualizations
    for ticker in missing_companies:
        sentiment_results.append({
            'ticker': ticker,
            'filing_date': None,
            'sentiment_compound': 0,
            'sentiment_positive': 0, 
            'sentiment_negative': 0,
            'sentiment_neutral': 0,
            'financial_sentiment_score': 0,
            'most_positive_sentence': "",
            'most_negative_sentence': ""
        })
    # Recreate our DataFrame with the missing companies added
    sentiment_df = pd.DataFrame(sentiment_results)
else:
    print(f"All {len(all_companies)} companies successfully processed")

# 4. CALCULATE RELATIVE SENTIMENT
# Find each company's average sentiment as a baseline
# This helps us understand if a filing is more positive/negative than the company's typical style
company_avg_sentiment = sentiment_df.groupby('ticker')['financial_sentiment_score'].mean().to_dict()

# Calculate how each filing compares to the company's average
# Positive values mean more positive than usual, negative means more negative than usual
sentiment_df['relative_sentiment'] = sentiment_df.apply(
    lambda row: row['financial_sentiment_score'] - company_avg_sentiment[row['ticker']], 
    axis=1
)

# FIXED SECTION: Remove duplicate rows with all zero sentiment values
sentiment_df = sentiment_df[~((sentiment_df['sentiment_compound'] == 0) & 
                             (sentiment_df['sentiment_positive'] == 0) & 
                             (sentiment_df['sentiment_negative'] == 0) &
                             (sentiment_df['sentiment_neutral'] == 0))]

# Save our enhanced sentiment data to a CSV file
enhanced_sentiment_path = os.path.join(SENTIMENT_OUTPUT_DIR, "enhanced_sentiment.csv")
sentiment_df.to_csv(enhanced_sentiment_path, index=False)
print(f"Saved enhanced sentiment data to {enhanced_sentiment_path}")

# 5. CREATE VISUALIZATIONS
# Show the distribution of sentiment scores across all companies
plt.figure(figsize=(12, 6))
plt.title("Distribution of Financial Sentiment Scores")
sns.histplot(sentiment_df['financial_sentiment_score'], kde=True)
plt.axvline(x=0, color='r', linestyle='--')  # Add a line at zero to show positive/negative split
dist_path = os.path.join(SENTIMENT_OUTPUT_DIR, "financial_sentiment_distribution.png")
plt.savefig(dist_path)
plt.close()

# Compare regular sentiment vs our financial sentiment to see how they differ
plt.figure(figsize=(12, 6))
plt.title("Comparison of Standard vs. Financial Sentiment")
plt.scatter(sentiment_df['sentiment_compound'], sentiment_df['financial_sentiment_score'], 
           alpha=0.6, c=sentiment_df['financial_sentiment_score'], cmap='coolwarm')
plt.xlabel("Standard VADER Sentiment")
plt.ylabel("Financial Sentiment Score")
plt.colorbar(label="Financial Sentiment")
plt.grid(True, linestyle='--', alpha=0.5)
comp_path = os.path.join(SENTIMENT_OUTPUT_DIR, "sentiment_comparison.png")
plt.savefig(comp_path)
plt.close()

# Show sentiment by company with error bars to visualize the range and consistency
plt.figure(figsize=(14, 7))
company_sentiment = sentiment_df.groupby('ticker')['financial_sentiment_score'].agg(['count', 'mean', 'min', 'max', 'std']).sort_values('mean')

# Fix companies with only one filing (they have no standard deviation)
company_sentiment['std'] = company_sentiment['std'].fillna(0)

# Plot the mean with error bars showing standard deviation
plt.errorbar(company_sentiment.index, company_sentiment['mean'], 
             yerr=company_sentiment['std'], fmt='o', capsize=5)

# Add lines showing the min-max range for each company
for i, ticker in enumerate(company_sentiment.index):
    plt.plot([i, i], [company_sentiment.loc[ticker, 'min'], company_sentiment.loc[ticker, 'max']], 
             'k-', alpha=0.3)

plt.axhline(y=0, color='gray', linestyle='--')  # Add a zero line
plt.title('Financial Sentiment by Company')
plt.ylabel('Financial Sentiment Score')
plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
company_sent_path = os.path.join(SENTIMENT_OUTPUT_DIR, "company_financial_sentiment.png")
plt.savefig(company_sent_path)
plt.close()

# Find the most common positive and negative words for each company
# This helps understand what's driving sentiment scores
company_pos_neg_words = {}

for ticker in df['ticker'].unique():
    # Combine all text for this company
    company_texts = " ".join(df[df['ticker'] == ticker]['processed_text'].dropna())

    # Handle empty text
    if not company_texts.strip():
        company_pos_neg_words[ticker] = {
            'positive': {'no_data': 0},
            'negative': {'no_data': 0}
        }
        continue

    # Break into words for counting
    words = word_tokenize(company_texts.lower())

    # Count financial terms in this company's text
    pos_words = {word: words.count(word) for word in financial_pos.keys() 
                 if words.count(word) > 0}
    neg_words = {word: words.count(word) for word in financial_neg.keys() 
                 if words.count(word) > 0}

    # Get the top 5 most frequent words in each category
    top_pos = dict(sorted(pos_words.items(), key=lambda x: x[1], reverse=True)[:5]) if pos_words else {'no_positive': 0}
    top_neg = dict(sorted(neg_words.items(), key=lambda x: x[1], reverse=True)[:5]) if neg_words else {'no_negative': 0}

    company_pos_neg_words[ticker] = {'positive': top_pos, 'negative': top_neg}

# Create a summary of sentiment terms for each company
sentiment_term_summary = []
for ticker, word_data in company_pos_neg_words.items():
    row = {'ticker': ticker}

    # Add top positive words
    for i, (word, count) in enumerate(word_data['positive'].items(), 1):
        row[f'top_pos_word_{i}'] = word
        row[f'top_pos_count_{i}'] = count

    # Add top negative words
    for i, (word, count) in enumerate(word_data['negative'].items(), 1):
        row[f'top_neg_word_{i}'] = word
        row[f'top_neg_count_{i}'] = count

    sentiment_term_summary.append(row)

# Save the word summary to a CSV file
term_summary_path = os.path.join(SENTIMENT_OUTPUT_DIR, "company_sentiment_terms.csv")
pd.DataFrame(sentiment_term_summary).to_csv(term_summary_path, index=False)

# Save the most positive and negative sentences for reference
# This helps put the numbers in context by showing actual examples
extreme_sentences_df = sentiment_df[['ticker', 'filing_date', 'most_positive_sentence', 'most_negative_sentence']]
extreme_sentences_df = extreme_sentences_df[
    (extreme_sentences_df['most_positive_sentence'] != "") | 
    (extreme_sentences_df['most_negative_sentence'] != "")
]
extreme_sentences_path = os.path.join(SENTIMENT_OUTPUT_DIR, "extreme_sentences.csv")
extreme_sentences_df.to_csv(extreme_sentences_path, index=False)

print(f"Enhanced sentiment analysis completed and saved to {SENTIMENT_OUTPUT_DIR}")
print(f"Companies analyzed: {len(company_sentiment)}")
print(f"Average financial sentiment by company:")
print(company_sentiment['mean'].sort_values(ascending=False))  # Sort from most positive to least

# Create charts for the executive summary
print("Creating visualizations needed for executive summary...")

# Bar chart of sentiment by company - useful for a quick overview
plt.figure(figsize=(12, 6))
plt.title("Enhanced Financial Sentiment by Company")
company_means = sentiment_df.groupby('ticker')['financial_sentiment_score'].mean().sort_values(ascending=False)
plt.bar(company_means.index, company_means.values, color='steelblue')
plt.axhline(y=0, color='red', linestyle='--')  # Add a line for zero
plt.ylabel('Financial Sentiment Score')
plt.tight_layout()
company_bar_path = os.path.join(SENTIMENT_OUTPUT_DIR, "enhanced_sentiment_by_company.png")
plt.savefig(company_bar_path)
plt.close()

# Grid of sentiment histograms for each company
# This shows the distribution of sentiment across all filings for each company
companies = sorted(sentiment_df['ticker'].unique())
n_companies = len(companies)
n_cols = 3  # Show 3 companies per row
n_rows = (n_companies + n_cols - 1) // n_cols  # Calculate how many rows we need

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()  # Make it easier to index

for i, ticker in enumerate(companies):
    company_data = sentiment_df[sentiment_df['ticker'] == ticker]['financial_sentiment_score']
    if len(company_data) > 0:
        sns.histplot(company_data, kde=True, ax=axes[i], color='skyblue')
        axes[i].set_title(f"{ticker} Sentiment Distribution")
        axes[i].set_xlabel('Financial Sentiment Score')
        axes[i].set_ylabel('Count')
        axes[i].axvline(x=0, color='red', linestyle='--')  # Add a line at zero

# Hide any empty plots at the end
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
hist_grid_path = os.path.join(SENTIMENT_OUTPUT_DIR, "sentiment_distribution_by_company.png")
plt.savefig(hist_grid_path)
plt.close()

print(f"Executive summary visualizations created successfully!")
print(f"All output saved to {SENTIMENT_OUTPUT_DIR}")


Performing enhanced sentiment analysis...
Current working directory: C:\Users\luke3\Documents\GitHub\Earnings Call Analyzer\notebooks
Cleaned transcripts file exists: True
Using data file: c:\users\luke3\documents\github\earnings call analyzer\data\cleaned_transcripts.csv (modified 2025-04-24 22:59:38)
Found 140 records with 23 companies
Companies in dataset: ['AAPL', 'AMZN', 'BAC', 'GOOGL', 'GS', 'HD', 'JNJ', 'JPM', 'KO', 'META', 'MRK', 'MSFT', 'NFLX', 'NVDA', 'PEP', 'PFE', 'PG', 'TSLA', 'UNH', 'V', 'VZ', 'WFC', 'WMT']
Available text columns for analysis: ['outlook_cleaned', 'product_mentions_cleaned']
All 23 companies successfully processed
Saved enhanced sentiment data to c:\users\luke3\documents\github\earnings call analyzer\data\output\enhanced_sentiment_viz\enhanced_sentiment.csv
Enhanced sentiment analysis completed and saved to c:\users\luke3\documents\github\earnings call analyzer\data\output\enhanced_sentiment_viz
Companies analyzed: 22
Average financial sentiment by company: